# DataPulse Guide AI — Project Tutor + Incident Investigator

This is a **specialized AI assistant powered by an existing open-weight Qwen model**, not a new foundation model trained from scratch. It has two grounded capabilities: (1) explain how DataPulse works from repository documentation, and (2) explain a real, synthetic incident report from DataPulse's deterministic engine. It is **independent of Gemini, Vertex AI and paid model API keys**. Optional local LLM inference needs model weights downloaded from Hugging Face.


## 1. Load the latest GitHub code

If possible, select **Runtime → Change runtime type → T4 GPU**. You can still run retrieval-only mode on CPU without downloading an LLM.


In [ ]:
from pathlib import Path
import subprocess, os
repo = Path('/content/aegis-real-time-supply-chain-ai')
if not (repo / '.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Codewithadi21/aegis-real-time-supply-chain-ai.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
os.chdir(repo/'datapulse-ai')
print('DataPulse root:',os.getcwd())


## 2. Install requirements and run tests

If you only want **free retrieval mode**, install `requirements.txt` without the local model requirements. For Qwen later, uncomment the second install line.


In [ ]:
%pip -q install -r requirements.txt
!python -m pytest -q
# %pip -q install -r requirements-local-llm.txt  # optional Qwen model


## 3. Build the real incident evidence (offline)


In [ ]:
from datapulse.replay import decision_replay
from datapulse.investigator import investigate
from datapulse.guide_assistant import answer_guide
summary, comparisons = decision_replay()
report = investigate(summary)
print('Verified incident:',report['incident_type'])
print('Prediction flips:',report['evidence']['prediction_flips'])


## 4. Project Guide mode — retrieval works without an LLM

The response shows exact source passages. This is transparent local **TF-IDF document retrieval**, not generation and not vector-embedding RAG.


In [ ]:
question = 'How does DataPulse work and why can schema validation miss semantic drift?'
result = answer_guide(question)
print(result['answer'])
print('\nSources:',result['sources'])


## 5. Incident Investigator mode — real evidence + documentation

This combines the retrieved documentation with the actual deterministic incident report, but still returns a source-attributed fallback without any model download.


In [ ]:
question = 'Why did the churn model predictions change and what should an engineer investigate?'
result = answer_guide(question, report=report)
print(result['answer'])
print('\nSources:',result['sources'])


## 6. Optional — load the Qwen LLM

This step downloads public model weights. No Google Cloud/Gemini API key is required. Free Colab T4 allocation is not guaranteed and CPU inference is slower. If memory is limited, change to `Qwen/Qwen2.5-0.5B-Instruct`.


In [ ]:
%pip -q install -r requirements-local-llm.txt
from datapulse.local_llm import load_local_model
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer, model = load_local_model(MODEL_ID)
print('Local model loaded:',MODEL_ID)


## 7. Chat naturally with your DataPulse project guide

The assistant provides source paths but generated text can still contain mistakes; compare it with the referenced Markdown documents and JSON report.


In [ ]:
from datapulse.guide_assistant import answer_guide
for q in ['Explain DataPulse from scratch for a beginner', 'Why did schema checks pass even with wrong feature meaning?', 'What limitations should I mention in a Google Data Engineer interview?']:
    result = answer_guide(q, tokenizer=tokenizer, model=model, report=report)
    print('\nQUESTION:',q)
    print('ANSWER:',result['answer'])
    print('SOURCES:',result['sources'])


## 8. Your own question

Edit the text and rerun the cell. The model and tokenizer stay loaded in the notebook runtime.


In [ ]:
my_question = 'What are the steps to verify DataPulse BigQuery integration?'
reply = answer_guide(my_question, tokenizer=tokenizer, model=model, report=report)
print(reply['answer'])
print('SOURCES:', reply['sources'])


## What this demonstration actually establishes

- **Built:** a specialized local assistant using a pretrained Qwen model, project-document retrieval, verified evidence and two question-answering modes
- **Not yet built:** custom-trained or fine-tuned LLM, hosted Vertex AI LLM endpoint, enterprise-grade autonomous agent, embedding-based retrieval, live data-source tools, or production-grade evaluation
- **Why it matters:** separate deterministic measurements from optional LLM explanations, and make project knowledge discoverable for users and interviewers

See `docs/DATAPULSE_USER_GUIDE.md` for a detailed walkthrough.
